In [1]:
from dotenv import load_dotenv  
load_dotenv()

True

In [22]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, GoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

In [4]:
loader=PyPDFLoader("../data/data_science_syllabus.pdf")
docs=loader.load()
len(docs)

10

In [5]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splitted_data=splitter.split_documents(docs)
len(splitted_data)

11

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

In [8]:
vector_store=Chroma.from_documents(documents=splitted_data, embedding=embeddings)

In [10]:
query="Machine learning and Data Science Content"
data=vector_store.similarity_search(query,k=3)  

In [ ]:
context=""
for doc in data:
    context+=doc.page_content+"\n"


In [18]:
llm=GoogleGenerativeAI(model="gemini-2.5-flash")


### Chain-Context_generate| Prompt | llm | strparser

In [25]:
def get_context(query:str):
    data=vector_store.similarity_search(query=query)
    context=""
    for doc in data:
        context+=doc.page_content+"\n"
    return {
        "context":context,
        "question":query
    }

In [23]:
prompt=PromptTemplate.from_template(
    """
    You are a helpful assistant and provide answer based on the context for user question and 
    if you don't know the answer, then you can say that 'I don't know' .
    context:{context}
    Question:{question}
    """
    
)

In [27]:
rag_chain=get_context | prompt | llm

In [31]:
res=rag_chain.invoke("What is My name")

In [32]:
print(res)

I don't know
